In [1]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import string

import re

import pdfplumber

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'SG MAS' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running SG MAS Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)






# %%

In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        regulatorName + ' 1': 'https://eservices.mas.gov.sg/fid/institution?sector=Banking',
        regulatorName + ' 2': 'https://eservices.mas.gov.sg/fid/institution?sector=Payments',
        regulatorName + ' 3': 'https://eservices.mas.gov.sg/fid/institution?sector=Capital%20Markets',
        regulatorName + ' 4': 'https://eservices.mas.gov.sg/fid/institution?sector=Insurance',
        regulatorName + ' 5': 'https://eservices.mas.gov.sg/fid/institution?sector=Financial%20Advisory',

        }



Typology={

       regulatorName + ' 1': 'List of Banking',
       regulatorName + ' 2': 'List of Payments',
       regulatorName + ' 3': 'List of Capital Markets',
       regulatorName + ' 4': 'List of Insurance',
       regulatorName + ' 5': 'List of Financial Advisory',

        }




sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 'RegCtry': [], 'RegCode' : [], 'ListCode': [], 
         'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 'Phone - Mother company': [], 'Check': []}




now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')




In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [6]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------
df_list = []

for reg in regdict:
    driver = webdriver.Chrome(options=chromeOptions)
    driver.maximize_window()
    
    print(f'Working with list {reg}')
    driver.get(regdict[reg])
    sleep(2)
    soup = BeautifulSoup(driver.page_source, 'html.parser')  
    sleep(5)

    # button = driver.find_element(By.XPATH, '//*[@id="spanResultPrint"]')
    # sleep(5)
    #button.click()
    inner_link = 'https://eservices.mas.gov.sg' + soup.find('span',id='spanResultPrint').find('a')['href']
    sleep(5)
    driver.quit()
    driver2 = webdriver.Chrome(options=chromeOptions)
    driver2.maximize_window()
    driver2.get(inner_link)
    sleep(5)
    # try:
    #     reference_element =  driver2.find_element(By.XPATH, '//*[@id="print-page-button"]')
    #     sleep(5)
    #     previous_button = reference_element.find_element(By.XPATH, 'preceding-sibling::button')

    # except Exception as e:
    #     raise e
    # sleep(2)
    # previous_button.click() 
    
    try:

        button_text = 'xls'
        button = driver.find_element(By.XPATH, f'//button[contains(text(), "{button_text}")]')
        button.click()
    except:
        driver2.refresh()
        sleep(30)
        button_text = 'xls'
        button = driver.find_element(By.XPATH, f'//button[contains(text(), "{button_text}")]')
        button.click()
    #driver.execute_script("arguments[0].click();", previous_button)       
    sleep(2)
    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
    sleep(10)
    dataframe = pd.read_csv(dl_files[0],on_bad_lines='skip' ,delimiter='\t')
    dataframe = dataframe.fillna('')
    sqldict = bourange_same_length_array(sqldict)   
    df1=pd.DataFrame(sqldict)
    
    df1['Name'] = dataframe['Organisation Name']
    df1['Address_1'] =  dataframe['Address']
    df1['Zip'] = dataframe['Address'].apply(lambda x: x.split(' ')[-1])
    df1['ListProcessDate'] = processdate
    df1['License_Type'] = dataframe['Licence Type/Status']
    df1['Phone'] = dataframe['Phone Number']
    df1['Website'] = dataframe['Website']
    df1['RegCtry'] = reg.split(' ')[0]
    df1['RegCode'] = reg.split(' ')[1]
    df1['ListCode'] = reg.split(' ')[-1]
    df1['ListName'] = Typology[reg]
    df1['RegulationType'] = 'Regulated' 
    df1.fillna('')
    df1 = df1.drop_duplicates()
    df_list.append(df1)
            
                

    
    if os.path.exists(tempfolder):

        for rem in os.listdir(tempfolder):

            os.remove(os.path.join(tempfolder, rem))
# driver.quit()

df = pd.concat(df_list, ignore_index=True)
    

    


Working with list SG MAS 1


MaxRetryError: HTTPConnectionPool(host='localhost', port=51413): Max retries exceeded with url: /session/e09affec1d2a417c00337b77481ef027/element (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002D59C44F1D0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

#df=pd.DataFrame(sqldict)

#df = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_21428\828776435.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df.to_csv('total_list5.csv')

In [ ]:
df1.drop_duplicates()

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,NaN,NaN,NaN,NaN,NaN,1291 ASIA PTE. LTD.,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,360 ONE CAPITAL PTE. LTD.,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,NaN,NaN,NaN,NaN,NaN,42 WEALTH MANAGEMENT PTE. LTD.,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,NaN,NaN,NaN,NaN,NaN,8 CAPITAL ASSET MANAGEMENT PTE LTD,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,NaN,NaN,NaN,NaN,NaN,ABA INSURANCE BROKERS PTE. LTD.,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3553,NaN,NaN,NaN,NaN,NaN,WYNNES FINANCIAL ADVISERS PTE. LTD.,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3556,NaN,NaN,NaN,NaN,NaN,XEN CAPITAL ASIA PTE. LTD. (CEASED ADVISING AN...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3558,NaN,NaN,NaN,NaN,NaN,YKN LUX INTERNATIONAL PTE. LTD.,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3560,NaN,NaN,NaN,NaN,NaN,ZHONGTAI INTERNATIONAL ASSET MANAGEMENT (SINGA...,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_combined.to_csv('total_.csv')

In [ ]:
pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.
